In [ ]:
from pathlib import Path
from typing import List, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

MISSING_SCORE = -1  # Missing data marker

# Paths
DATA_DIR = Path("/Users/rachelmorse/Documents/2023:2024/Data")
TP1_FILE = DATA_DIR / "BBHI-Senior" / "bbhi senior data.xlsx"
TP2_FILE = DATA_DIR / "BBHI-Senior" / "Ministerio2024Wave2_DATA_2026-01-20_1245.csv"
EXPORT_DIR = DATA_DIR / "Exported data"

In [ ]:
# First convert the date columns to strings because they are causing trouble
columns_to_string = ["birthdate", "date_NPS_TP5"]
dtype_spec = {col: str for col in columns_to_string}

# Read in BBHI Senior tp 1 data
sa_tp1 = pd.read_excel(TP1_FILE, dtype=dtype_spec)

print(f"Number of participants: {len(sa_tp1)}")

In [ ]:
# Rename variable in BBHI Senior df so they match BBHI df
filtered_sa_df = sa_tp1.rename(
    columns={
        "ID": "id",
        "RAVLT_delayed": "w1_delayed_recall_raw",
        "TMTB_sec": "w1_tmt_b_raw",
        "Sem_fluency": "w1_sem_fluency_raw",
        "Digit_span_back": "w1_inverse_digits_raw",
        "RAVLT_total_learn": "w1_ravlt_total",
    }
)

In [ ]:
# Function to calculate exact ages
def parse_date(date_str: Union[str, float]) -> pd.Timestamp:
    """Parse date string based on criteria below.

    They are formatting oddly out of excel where they are both
    American and British format so each of these formats
    needs to be treated differently.

    Args:
        date_str: Date string from excel.

    Returns:
        Parsed datetime object or NaT if invalid.
    """
    if pd.isna(date_str):
        return pd.NaT
    s = str(date_str).strip()

    if "00:00:00" in s:
        # Treat as yyyy-dd-mm format
        return pd.to_datetime(s, format="%Y-%d-%m 00:00:00", errors="coerce")
    # Treat as dd/mm/yyyy format
    return pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")


def preprocess_dates(df: pd.DataFrame, birth_col: str, collect_col: str) -> None:
    """Convert date columns to datetime and drop rows with invalid dates.

    Args:
        df: DataFrame containing the date columns.
        birth_col: Name of the birthdate column.
        collect_col: Name of the data collection date column.
    """
    df[birth_col] = df[birth_col].apply(parse_date)
    df[collect_col] = df[collect_col].apply(parse_date)
    df.dropna(subset=[birth_col, collect_col], inplace=True)


def calculate_age(birth_date: pd.Timestamp, collection_date: pd.Timestamp) -> float:
    """Calculate the exact age based on birthdate and collection date.

    Args:
        birth_date: Birthdate.
        collection_date: Date of data collection.

    Returns:
        Exact age of subs at collection date.
    """
    # Calculate difference in days
    days_difference = (collection_date - birth_date).days

    # Convert days to years using the average number of days in a year
    age = days_difference / 365.25
    return age


# Preprocess date columns in the DataFrame
preprocess_dates(filtered_sa_df, "birthdate", "date_NPS_TP5")

# Calculate and add age to DataFrame
filtered_sa_df["w1_age"] = filtered_sa_df.apply(
    lambda row: calculate_age(row["birthdate"], row["date_NPS_TP5"]), axis=1
)

# Display the first few rows to verify results
filtered_sa_df.sample(10)[["id", "NP_age", "w1_age", "birthdate", "date_NPS_TP5"]].head(10)

In [ ]:
# Filter df to only include needed columns
filtered_sa_df = filtered_sa_df[
    [
        "id",
        "w1_age",
        "YoE",
        "sex",
        "w1_delayed_recall_raw",
        "w1_tmt_b_raw",
        "w1_sem_fluency_raw",
        "w1_inverse_digits_raw",
        "w1_ravlt_total",
    ]
]

In [ ]:
# Read in BBHI Senior tp 2 data
sa_tp2 = pd.read_csv(TP2_FILE)

print(f"Number of participants: {len(sa_tp2)}")

In [ ]:
# Rename variable in so they match BBHI df
filtered_sa_tp2 = sa_tp2.rename(
    columns={
        "record_id_np_w2": "id",
        "animals_w2": "w2_sem_fluency_raw",
        "ravlt_delayed_w2": "w2_delayed_recall_raw",
        "tmtb_w2": "w2_tmt_b_raw",
        "digits_backwards_w2": "w2_inverse_digits_raw",
        "ravlt_total_w2": "w2_ravlt_total",
    }
)

In [ ]:
# Calculate the exact ages
def preprocess_dates_t2(df: pd.DataFrame, birth_col: str, collect_col: str) -> pd.DataFrame:
    """Convert date columns to datetime.

    Args:
        df: DataFrame containing the date columns.
        birth_col: Name of the birthdate column.
        collect_col: Name of the data collection date column.

    Returns:
        DataFrame with converted date columns.
    """
    # Check if the dates are already datetime objects
    if not pd.api.types.is_datetime64_any_dtype(df[birth_col]):
        df[birth_col] = pd.to_datetime(df[birth_col], errors="coerce")

    if not pd.api.types.is_datetime64_any_dtype(df[collect_col]):
        df[collect_col] = pd.to_datetime(df[collect_col], errors="coerce")

    return df


# Call the function
filtered_sa_tp2 = preprocess_dates_t2(filtered_sa_tp2, "birthdate", "date_np_wave2")

# Calculate and add age to DataFrame, only when both dates are present
filtered_sa_tp2["w2_age"] = filtered_sa_tp2.apply(
    lambda row: calculate_age(row["birthdate"], row["date_np_wave2"])
    if pd.notna(row["birthdate"]) and pd.notna(row["date_np_wave2"])
    else pd.NA,
    axis=1,
)

# Display the first few rows to verify results
filtered_sa_tp2[["id", "age", "w2_age", "birthdate", "date_np_wave2"]].head()

In [ ]:
# Select only the required columns
filtered_sa_tp2 = filtered_sa_tp2[
    [
        "id",
        "w2_age",
        "w2_delayed_recall_raw",
        "w2_sem_fluency_raw",
        "w2_tmt_b_raw",
        "w2_inverse_digits_raw",
        "w2_ravlt_total",
    ]
]

In [ ]:
# Merge dfs by participant ID for tp1 and tp2
merged_df = pd.merge(filtered_sa_df, filtered_sa_tp2, on="id", how="inner")
merged_df_tp1 = pd.merge(filtered_sa_df, filtered_sa_tp2, on="id", how="outer")

print(f"Number of participants: {len(merged_df)}")
print(f"Number of participants at tp1: {len(merged_df_tp1)}")

merged_df.head()

In [ ]:
# Drop participants with missing neuropsych data and -1 values in the specific columns

# Get the number of rows before dropping participants
rows_before = merged_df.shape[0]
rows_before_tp1 = merged_df_tp1.shape[0]

columns = [
    "YoE",  # years of education
    "w1_age",  # age tp1
    "w2_age",  # age tp2
    "w1_delayed_recall_raw",  # RAVLT tp1
    "w1_sem_fluency_raw",  # Semantic fluency tp1
    "w1_tmt_b_raw",  # TMT B tp1
    "w1_inverse_digits_raw",  # Digit span backward tp1
    "w2_delayed_recall_raw",  # RAVLT tp2
    "w2_sem_fluency_raw",  # Semantic fluency tp2
    "w2_tmt_b_raw",  # TMT B tp2
    "w2_inverse_digits_raw",  # Digit span backward tp2
    "w1_ravlt_total",  # RAVLT total tp1
    "w2_ravlt_total",  # RAVLT total tp2
]

columns_tp1 = [
    "YoE",
    "w1_age",
    "w1_delayed_recall_raw",
    "w1_ravlt_total",
    "w1_sem_fluency_raw",
    "w1_tmt_b_raw",
    "w1_inverse_digits_raw",
]

# Drop participants with missing neuropsych data
merged_df.dropna(subset=columns, inplace=True)
merged_df_tp1.dropna(subset=columns_tp1, inplace=True)

# Replace -1 values in specific columns with NA
for col in columns:
    merged_df[col] = merged_df[col].replace(MISSING_SCORE, np.nan)
    # Ive put all cols for tp1 because this list gets used to make the superagers list
    merged_df_tp1[col] = merged_df_tp1[col].replace(MISSING_SCORE, np.nan)

# Drop participant with 0 as their age
merged_df = merged_df[merged_df["w1_age"] != 0]
merged_df_tp1 = merged_df_tp1[merged_df_tp1["w1_age"] != 0]

# Get the number of rows after dropping participants
rows_after = merged_df.shape[0]
rows_after_tp1 = merged_df_tp1.shape[0]

# Calculate the number of participants dropped
participants_dropped = rows_before - rows_after
participants_dropped_tp1 = rows_before_tp1 - rows_after_tp1

print(f"Number of participants dropped longitudinally: {participants_dropped}")
print(f"Number of participants dropped at tp1: {participants_dropped_tp1}")

In [ ]:
# NOTE there are no participants who need NPS data excluded due to invalid data from NPS comments


def plot_score_distributions(df: pd.DataFrame, cols: List[str], label_step: int = 4) -> None:
    """Plot the count of each score to check for participants who need to be dropped.

    Args:
        df: Neuropsych data.
        cols: Score columns to plot.
        label_step: Label the count on every nth bar.
    """
    for col in cols:
        plt.figure(figsize=(16, 5))  # Make the figure wider
        ax = sns.countplot(data=df, x=col)
        plt.title(f"Bar Distribution of {col}")
        plt.xlabel(col)
        plt.ylabel("Count")

        for i, p in enumerate(ax.patches):
            if i % label_step == 0:
                count = round(p.get_height())
                ax.annotate(
                    str(count),
                    (p.get_x() + p.get_width() / 2.0, p.get_height()),
                    ha="center",
                    va="bottom",
                    fontsize=10,
                    color="black",
                )
        plt.show()


# Plot the data to make sure there are no other people who need to be dropped
neuropsych_cols = [
    "w1_delayed_recall_raw",
    "w2_delayed_recall_raw",
    "w1_ravlt_total",
    "w2_ravlt_total",
    "w1_tmt_b_raw",
    "w2_tmt_b_raw",
    "w1_sem_fluency_raw",
    "w2_sem_fluency_raw",
    "w1_inverse_digits_raw",
    "w2_inverse_digits_raw",
]
plot_score_distributions(merged_df, neuropsych_cols)

In [ ]:
# Replace 1 with 'male' and 2 with 'female' in the 'sex' column
merged_df["sex"] = merged_df["sex"].replace({1: "male", 2: "female"})
merged_df_tp1["sex"] = merged_df_tp1["sex"].replace({1: "male", 2: "female"})

merged_df[["id", "sex"]].head()

In [ ]:
def print_age_summary(df: pd.DataFrame) -> None:
    """Print the number of participants and the mean, SD, and range of baseline age.

    Args:
        df: Participant data with a w1_age column.
    """
    ages = df["w1_age"]
    print(f"Number of participants: {ages.count()}")
    print(f"Mean age: {ages.mean():.2f}")
    print(f"SD age: {ages.std():.2f}")
    print(f"Range age: {ages.min():.2f} - {ages.max():.2f}")


# Look at age data for all available cross-sectional data (merged cohort data)
print_age_summary(merged_df_tp1)

In [ ]:
# Look at age data for all available longitudinal data (merged cohort data)
print_age_summary(merged_df)

In [ ]:
# Export this df to a csv to use for future analysis
merged_df.to_csv(EXPORT_DIR / "clean_bbhi_senior.csv", index=False)
merged_df_tp1.to_csv(EXPORT_DIR / "clean_bbhi_senior_tp1.csv", index=False)